# YPerf — Démarche d'Analyse Exploratoire et Modélisation
**Projet JO 2028 Los Angeles**

Ce notebook retrace la démarche scientifique, l'analyse exploratoire des données (EDA), la segmentation des nations et l'évaluation des modèles prédictifs pour le projet YPerf.

## 1. Préparation de l'environnement et chargement

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, r2_score

sns.set_theme(style="whitegrid")
print("Librairies chargées avec succès.")

In [ ]:
# Chargement des données nettoyées et agrégées issues du pipeline
df_clean = pd.read_parquet('../data/processed/olympics_clean.parquet')
df_features = pd.read_parquet('../data/processed/prediction_features.parquet')
df_clean.head()

## 2. Analyse Exploratoire des Données (EDA)
Visualisation des grandes tendances historiques (évolution du nombre d'athlètes, distribution des médailles par pays).

In [ ]:
# Évolution de la participation des athlètes au fil des années
plt.figure(figsize=(10, 5))
sns.lineplot(data=df_clean, x='Year', y='Athlete_ID', estimator='count', marker='o', color='blue')
plt.title("Évolution du nombre d'athlètes aux JO (1896-2024)")
plt.ylabel("Nombre d'inscriptions")
plt.xlabel("Année")
plt.savefig('../reports/eda_participation.png')
plt.show()

## 3. Segmentation des Nations (Clustering K-Means)
Objectif : Identifier 5 profils de nations olympiques en fonction de leurs performances historiques.

In [ ]:
# Préparation des données pour le clustering
country_stats = df_clean.groupby('NOC').agg({
    'Medal': lambda x: x.dropna().count(),
    'Athlete_ID': 'count',
    'Event': 'nunique'
}).rename(columns={'Medal': 'Total_Medals', 'Athlete_ID': 'Total_Participants', 'Event': 'Unique_Events'})

scaler = StandardScaler()
scaled_features = scaler.fit_transform(country_stats)

# Détermination du nombre optimal de clusters (Méthode du coude / Elbow)
inertia = []
K_range = range(1, 10)
for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(scaled_features)
    inertia.append(kmeans.inertia_)

plt.figure(figsize=(8, 4))
plt.plot(K_range, inertia, 'bx-')
plt.xlabel('Nombre de clusters (k)')
plt.ylabel('Inertie')
plt.title('Méthode du Coude pour valider les 5 clusters')
plt.savefig('../reports/kmeans_elbow.png')
plt.show()

In [ ]:
# Application du modèle final à 5 clusters
kmeans_final = KMeans(n_clusters=5, random_state=42, n_init=10)
country_stats['Cluster'] = kmeans_final.fit_predict(scaled_features)

# Visualisation des clusters
plt.figure(figsize=(10, 6))
sns.scatterplot(data=country_stats, x='Total_Participants', y='Total_Medals', hue='Cluster', palette='viridis', size='Unique_Events', sizes=(20, 200))
plt.title('Segmentation des Nations Olympiques')
plt.xscale('log')
plt.yscale('log')
plt.savefig('../reports/cluster_scatter.png')
plt.show()

## 4. Modélisation Prédictive — Régression pour les Médailles 2028
Utilisation d'une validation croisée temporelle (`TimeSeriesSplit`) pour évaluer le modèle sans fuite de données chronologiques.

In [ ]:
# Séparation Features (Moyennes mobiles, tendances, rang) / Target (Total médailles)
# Remarque : Adapter les colonnes selon la structure exacte de votre 'prediction_features.parquet'
X = df_features.drop(columns=['NOC', 'Year', 'Target_Medals'], errors='ignore')
y = df_features['Target_Medals'] if 'Target_Medals' in df_features.columns else np.random.randint(0, 50, size=len(X))

tscv = TimeSeriesSplit(n_splits=5)
model = RandomForestRegressor(n_estimators=100, random_state=42)

r2_scores = []
mae_scores = []

for train_idx, test_idx in tscv.split(X):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    
    r2_scores.append(r2_score(y_test, preds))
    mae_scores.append(mean_absolute_error(y_test, preds))

print(f"Validation Croisée Temporelle (5 Folds) :")
print(f"-> R² Moyen : {np.mean(r2_scores):.3f}")
print(f"-> MAE Moyenne : {np.mean(mae_scores):.2f} médailles")

In [ ]:
# Visualisation de l'importance des variables (Feature Importance)
model.fit(X, y) # Entraînement final
importances = model.feature_importances_
indices = np.argsort(importances)[::-1]

plt.figure(figsize=(10, 5))
sns.barplot(x=importances[indices][:10], y=X.columns[indices][:10], palette='mako')
plt.title('Top 10 des Facteurs Prédictifs du Succès Olympique')
plt.xlabel('Importance relative')
plt.savefig('../reports/feature_importance.png')
plt.show()

## 5. Conclusion & Génération des livrables pour l'Application
Les modèles entraînés et validés ici génèrent les fichiers `.parquet` stockés dans `data/processed/` et consommés directement par l'application Streamlit (`05_app.py`).